# Spherical Magnetic Locator

This notebook trains a physics-biased conditional density model for the endpoint of a noisy magnetic trajectory, and estimates a mutual-information lower bound

$$I(\text{endpoint}; \text{trajectory}) \gtrsim h(\text{endpoint}) - h(\text{endpoint}\mid\text{trajectory}).$$

## 1. Setup

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.nn import functional as F

NOTEBOOK_ROOT = Path.cwd()
if (NOTEBOOK_ROOT / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT))
elif (NOTEBOOK_ROOT / "notebooks" / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT / "notebooks"))

from shared.magentic_field import (
    DEVICE,
    DTYPE,
    EARTH_RADIUS_M,
    STEP_METERS,
    STEP_RAD,
    exp_map_sphere,
    normalize,
    tangent_basis,
)
from shared.trajectory_sampler import (
    FEATURE_DIM,
    make_trajectories,
    estimate_feature_normalization,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device={DEVICE}, step={STEP_METERS:.1f} m, angular step={STEP_RAD:.3e} rad")

## 2. Conditional Density Model

In [ ]:
FEATURE_MEAN, FEATURE_STD = estimate_feature_normalization(device=DEVICE)

N_COMPONENTS = 8
D_MODEL = 384
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.03
TRAIN_EPOCHS = 480
TRAIN_BATCHES_PER_EPOCH = 8
TRAIN_BATCH_SIZE = 512
LEARNING_RATE = 2e-4
DIRECTION_PRETRAIN_EPOCHS = 80
AUX_DIRECTION_WEIGHT = 2.0
PLOT_EVERY = 10

OFFSET_SCALE_M = 2_000_000.0
OFFSET_INITIAL_RADIUS_M = 350_000.0
SIGMA_MIN_M = 10.0
SIGMA_MAX_M = math.pi * EARTH_RADIUS_M
SIGMA_INITIAL_M = 500_000.0
LOG_SIGMA_MIN = math.log(SIGMA_MIN_M)
LOG_SIGMA_MAX = math.log(SIGMA_MAX_M)
SIGMA_INITIAL_UNIT = (math.log(SIGMA_INITIAL_M) - LOG_SIGMA_MIN) / (
    LOG_SIGMA_MAX - LOG_SIGMA_MIN
)
SIGMA_INITIAL_LOGIT = math.log(SIGMA_INITIAL_UNIT / (1.0 - SIGMA_INITIAL_UNIT))

EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
EVAL_BATCH_SIZE = 1024
EVAL_BATCHES_PER_POINT = 4


class EndpointTangentGaussianSequenceModel(nn.Module):
    def __init__(self, n_components=N_COMPONENTS, d_model=D_MODEL, n_layers=N_LAYERS):
        super().__init__()
        self.n_components = n_components
        self.register_buffer("feature_mean", FEATURE_MEAN.detach().clone())
        self.register_buffer("feature_std", FEATURE_STD.detach().clone())
        self.input_projection = nn.Sequential(
            nn.Linear(FEATURE_DIM, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
        )
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=DROPOUT if n_layers > 1 else 0.0,
        )
        self.attention = nn.Linear(2 * d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(2 * d_model, n_components + n_components * 4),
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)
        with torch.no_grad():
            angles = (
                2.0 * math.pi * torch.arange(n_components, dtype=DTYPE) / n_components
            )
            radius = OFFSET_INITIAL_RADIUS_M / OFFSET_SCALE_M
            offset_bias = torch.stack(
                [radius * torch.cos(angles), radius * torch.sin(angles)], dim=-1
            ).clamp(-0.95, 0.95)
            self.head[-1].bias[n_components:].view(n_components, 4)[:, :2].copy_(
                torch.atanh(offset_bias)
            )

    def forward(self, x, pad_mask):
        scaled = (x - self.feature_mean) / self.feature_std
        tokens = self.input_projection(scaled)
        encoded, _ = self.gru(tokens)
        attention_logits = self.attention(encoded).squeeze(-1)
        attention_logits = attention_logits.masked_fill(pad_mask, -torch.inf)
        weights = torch.softmax(attention_logits, dim=-1)
        pooled = torch.sum(encoded * weights[:, :, None], dim=1)
        raw = self.head(pooled)

        logits = raw[:, : self.n_components]
        comp = raw[:, self.n_components :].reshape(-1, self.n_components, 4)
        last_idx = (~pad_mask).sum(dim=1).sub(1).clamp_min(0)
        last_B = x[torch.arange(x.shape[0], device=x.device), last_idx, :3]
        anchor = normalize(last_B)
        e1, e2 = tangent_basis(anchor)
        offset_m = OFFSET_SCALE_M * torch.tanh(comp[..., :2])
        tangent_step_m = (
            offset_m[..., 0:1] * e1[:, None, :] + offset_m[..., 1:2] * e2[:, None, :]
        )
        mean_direction = exp_map_sphere(anchor[:, None, :], tangent_step_m)

        sigma_unit = torch.sigmoid(comp[..., 2:] + SIGMA_INITIAL_LOGIT)
        log_sigma_m = LOG_SIGMA_MIN + (LOG_SIGMA_MAX - LOG_SIGMA_MIN) * sigma_unit
        return logits, mean_direction, log_sigma_m


def log_map_sphere_meters(base, target):
    y = target[:, None, :].expand_as(base)
    dot = (y * base).sum(dim=-1).clamp(-1.0, 1.0)
    tangent = y - dot[..., None] * base
    sin_theta = tangent.norm(dim=-1).clamp_min(1e-12)
    theta = torch.atan2(sin_theta, dot)
    direction = tangent / sin_theta[..., None]
    e1, e2 = tangent_basis(base)
    v_m = EARTH_RADIUS_M * theta[..., None] * direction
    coords = torch.stack([(v_m * e1).sum(dim=-1), (v_m * e2).sum(dim=-1)], dim=-1)
    log_area_jac = 2.0 * math.log(EARTH_RADIUS_M) + torch.log(
        (theta / sin_theta).clamp_min(1e-12)
    )
    small = theta < 1e-5
    log_area_jac = torch.where(
        small,
        torch.full_like(log_area_jac, 2.0 * math.log(EARTH_RADIUS_M)),
        log_area_jac,
    )
    return coords, log_area_jac


def tangent_gaussian_mixture_log_prob(endpoint, logits, mean_direction, log_sigma_m):
    coords_m, log_area_jac = log_map_sphere_meters(mean_direction, endpoint)
    inv_sigma = torch.exp(-log_sigma_m)
    whitened = coords_m * inv_sigma
    log_component = (
        -0.5 * (whitened.square().sum(dim=-1) + 2.0 * math.log(2.0 * math.pi))
        - log_sigma_m.sum(dim=-1)
        + log_area_jac
    )
    return torch.logsumexp(F.log_softmax(logits, dim=-1) + log_component, dim=-1)


def nll_loss(model, x, pad_mask, endpoint):
    params = model(x, pad_mask)
    return -tangent_gaussian_mixture_log_prob(endpoint, *params).mean()


def mixture_mean_direction(logits, mean_direction):
    weights = F.softmax(logits, dim=-1)
    return normalize((weights[..., None] * mean_direction).sum(dim=1))


def training_objective(model, x, pad_mask, endpoint, direction_only=False):
    params = model(x, pad_mask)
    logits, mean_direction, _ = params
    nll = -tangent_gaussian_mixture_log_prob(endpoint, *params).mean()
    best_dot = (mean_direction * endpoint[:, None, :]).sum(dim=-1).max(dim=-1).values
    pred_dir = mixture_mean_direction(logits, mean_direction)
    mean_dot = (pred_dir * endpoint).sum(dim=-1)
    aux_direction = 0.7 * (1.0 - best_dot).mean() + 0.3 * (1.0 - mean_dot).mean()
    if direction_only:
        return aux_direction, nll.detach(), aux_direction
    return nll + AUX_DIRECTION_WEIGHT * aux_direction, nll, aux_direction


def create_model_bundle():
    model = EndpointTangentGaussianSequenceModel().to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_EPOCHS * TRAIN_BATCHES_PER_EPOCH,
        eta_min=0.1 * LEARNING_RATE,
    )
    return model, optimizer, scheduler


_parameter_count_model = EndpointTangentGaussianSequenceModel().to(
    device=DEVICE, dtype=DTYPE
)
sum(p.numel() for p in _parameter_count_model.parameters())

In [ ]:
def train_model_for_steps(step_count):
    model, optimizer, scheduler = create_model_bundle()
    history = []
    model.train()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        epoch_objectives = []
        epoch_nlls = []
        epoch_aux = []
        for _ in range(TRAIN_BATCHES_PER_EPOCH):
            x, pad_mask, endpoint, _ = make_trajectories(
                TRAIN_BATCH_SIZE, step_counts=step_count
            )
            direction_only = epoch <= DIRECTION_PRETRAIN_EPOCHS
            loss, nll, aux_direction = training_objective(
                model, x, pad_mask, endpoint, direction_only=direction_only
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_objectives.append(float(loss.detach().cpu()))
            epoch_nlls.append(float(nll.detach().cpu()))
            epoch_aux.append(float(aux_direction.detach().cpu()))

        row = dict(
            epoch=epoch,
            objective=float(np.mean(epoch_objectives)),
            nll=float(np.mean(epoch_nlls)),
            nll_bits=float(np.mean(epoch_nlls) / math.log(2.0)),
            aux_direction=float(np.mean(epoch_aux)),
            phase=(
                "direction pretrain"
                if epoch <= DIRECTION_PRETRAIN_EPOCHS
                else "density NLL"
            ),
        )
        history.append(row)
        if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == TRAIN_EPOCHS:
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["nll_bits"] for item in history],
                    mode="lines",
                    name=f"steps={step_count} train NLL",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["aux_direction"] for item in history],
                    mode="lines",
                    name="direction aux",
                    yaxis="y2",
                )
            )
            fig.update_layout(
                title=f"Training fixed-step tangent Gaussian locator, steps={step_count}, epoch {epoch}/{TRAIN_EPOCHS}, {row['phase']}",
                xaxis_title="epoch",
                yaxis_title="NLL = estimated h(endpoint | trajectory), bits",
                yaxis2=dict(title="direction aux", overlaying="y", side="right"),
                height=360,
                margin=dict(l=40, r=20, t=50, b=40),
            )
            display(fig)

    return {"model": model, "history": history, "step_count": step_count}


trained_models = {}
for steps in EVAL_STEP_COUNTS:
    trained_models[int(steps)] = train_model_for_steps(int(steps))

for steps, result in trained_models.items():
    print(
        f"steps={steps:2d}: final train NLL={result['history'][-1]['nll_bits']:.4f} bits"
    )

## 3. Evaluation

In [ ]:
@torch.no_grad()
def estimate_conditional_entropy(model, step_count):
    model.eval()
    weighted_loss = 0.0
    total = 0
    for _ in range(EVAL_BATCHES_PER_POINT):
        x, pad_mask, endpoint, _ = make_trajectories(
            EVAL_BATCH_SIZE, step_counts=step_count
        )
        loss = nll_loss(model, x, pad_mask, endpoint)
        weighted_loss += float(loss.cpu()) * EVAL_BATCH_SIZE
        total += EVAL_BATCH_SIZE
    return weighted_loss / total


h_endpoint_unit_bits = math.log2(4.0 * math.pi)
h_endpoint_earth_surface_bits = math.log2(4.0 * math.pi * EARTH_RADIUS_M**2)
print(f"h(endpoint) on unit sphere: {h_endpoint_unit_bits:.4f} bits")
print(
    f"h(endpoint) on Earth surface: {h_endpoint_earth_surface_bits:.4f} bits; MI is unchanged by this area-scale shift"
)

rows = []
for trained_steps, result in trained_models.items():
    model = result["model"]
    for eval_steps in EVAL_STEP_COUNTS:
        h_cond_bits = estimate_conditional_entropy(model, int(eval_steps)) / math.log(
            2.0
        )
        info_bits = h_endpoint_unit_bits - h_cond_bits
        rows.append(
            dict(
                trained_steps=int(trained_steps),
                eval_steps=int(eval_steps),
                distance_m=int(eval_steps) * STEP_METERS,
                h_cond_bits=h_cond_bits,
                info_bits=info_bits,
            )
        )

for row in rows:
    print(
        f"trained={row['trained_steps']:2d}, eval={row['eval_steps']:2d}, "
        f"distance={row['distance_m']:6.1f} m, "
        f"h_cond={row['h_cond_bits']:.4f} bits, I_lb={row['info_bits']:.4f} bits"
    )

In [ ]:
fig = go.Figure()
for trained_steps in sorted({row["trained_steps"] for row in rows}):
    curve = sorted(
        [row for row in rows if row["trained_steps"] == trained_steps],
        key=lambda row: row["eval_steps"],
    )
    fig.add_trace(
        go.Scatter(
            x=[row["eval_steps"] for row in curve],
            y=[row["info_bits"] for row in curve],
            mode="lines+markers",
            name=f"trained steps={trained_steps}",
            customdata=[[row["h_cond_bits"]] for row in curve],
            hovertemplate=(
                "trained steps="
                + str(trained_steps)
                + "<br>eval steps=%{x}<br>I=%{y:.4f} bits<br>h_cond=%{customdata[0]:.4f} bits<extra></extra>"
            ),
        )
    )
fig.update_layout(
    title="Cross-step I(endpoint; trajectory) lower bound by trained fixed-step density model",
    xaxis=dict(title="evaluation steps", dtick=1),
    yaxis=dict(title="lower bound, bits"),
    height=470,
    margin=dict(l=50, r=55, t=50, b=45),
)
fig.show()